# CRIBA — reproducible ideation (60-second demo)

CRIBA is a deterministic, local-first ideation engine. The same seed and the
same catalog always produce the same ideas, on any machine, with no network
and no API key. Optionally, ideas can be expanded with **free** cloud models
(Z.ai `glm-5.3-flash` and Nous `poolside/laguna-s-2.1:free`).

```bash
pip install criba
```

In [ ]:
from criba.catalog import methods
from criba.lottery import LotteryEngine

QUERY = "how can we design secure approvals for autonomous agents?"


def run_once(seed: int = 42):
    engine = LotteryEngine.from_methods(methods(), seed=seed)
    engine.run_round(mode="alternating", batch_size=8, query=QUERY)
    engine.run_round(mode="alternating", batch_size=8, query=QUERY)
    return engine.get_top_ideas(5)


first = run_once(seed=42)
for i, idea in enumerate(first, 1):
    print(f"{i:2d}. [{idea['quality']:>13}] score {idea['score']:.3f}  {idea['title'][:66]}")

# Reproducibility: a second engine with the same seed yields identical ideas.
second = run_once(seed=42)
assert [(i\["title"\], i\["score"\]) for i in first] == \
       [(i\["title"\], i\["score"\]) for i in second]
print("[ok] Two engines, same seed, same ideas — reproducible.")

## What just happened

- Round 1 (associative) matched methods to your query; round 2 (pure) sampled
  the frozen catalog randomly — both seeded.
- `LotteryEngine.from_methods(catalog, seed)` is deliberately public and
  side-effect free, so you can embed it in your own pipelines.

## Next steps

```bash
criba --help
criba run --query "reduce cold-store energy in data centers" --json
criba blackforge --query "mitigate prompt injection in approval flows" --seed 1
criba serve   # loopback API, Swagger at /docs
criba gui     # desktop
```

Check the audit trail in the default SQLite store
(`artifacts/criba.sqlite3` from a source checkout, or `%LOCALAPPDATA%\CRIBA-Blackforge`
for pip/portable installs).